# Flow types

Adapted from the [summer2 documentation](https://summer2.readthedocs.io)
page `examples/04-flow-types` at commit
`d1537d6188aba85c33c0449b197eef6ad8b03d6c` of
[monash-emu/summer2](https://github.com/monash-emu/summer2).

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

summer4 exposes three declarative flow classes — `TransitionFlow`,
`ExitFlow`, and `EntryFlow` — plus infection via `ForceOfInfection` on `FlowModel`. This page ports each summer2 constructor example into that idiom.


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    EntryFlow,
    Everything,
    ExitFlow,
    FlowMass,
    FlowModel,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Time,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix
from summer4.timevarying import piecewise

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

state = Property("state", ("S", "I", "R"))
pop = Property("pop", ("all",))
pmap = PropertyMap.from_property(state).stratify(pop)


def wrap_y0(arr):
    return PropertyData.wrap(pmap, jnp.asarray(arr, dtype=jnp.float64))


def run(model, y0, t1=20.0, extra=None, params=None):
    req = {"comp": SaveRequest(Compartments())}
    if extra:
        req.update(extra)
    plan = SavePlan(requests=req, ts=jnp.linspace(0.0, t1, 201))
    if not isinstance(y0, PropertyData):
        y0 = wrap_y0(y0)
    return model.compile().run(
        params or {}, y0, t0=0.0, t1=t1, dt=0.1, save=plan, solver="euler"
    )


def plot_comp(res, title):
    return res["comp"].to_pandas().plot(
        title=title, labels={"index": "time (days)", "value": "people"}
    )


## Transition flow

A fractional transition: 10% of infectious people recover per day.


In [ ]:
m = FlowModel(pmap)
m.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.1))
y0 = jnp.zeros(pmap.size)
y0 = y0.at[pmap.select(state["I"])].set(1000.0)
res = run(m, wrap_y0(y0))
r_final = float(np.asarray(res["comp"].select(state["R"]).values.data)[-1, 0])
assert r_final > 500.0
plot_comp(res, "Transition flow: recovery")


## Time-varying transition rate

Any rate may be a time-varying expression. After day 10 recovery jumps from
0.1 to 0.4 (`piecewise` / summer2's `get_piecewise_scalar_function`).


In [ ]:
recovery_rate = piecewise(Time(), (10.0,), (0.1, 0.4))
m = FlowModel(pmap)
m.add_flow(TransitionFlow("recovery", state["I"], state["R"], recovery_rate))
y0 = jnp.zeros(pmap.size)
y0 = y0.at[pmap.select(state["I"])].set(1000.0)
res = run(m, wrap_y0(y0))
# Faster recovery after day 10 empties I more quickly than a constant 0.1.
i_mid = float(np.asarray(res["comp"].select(state["I"]).values.data)[100, 0])
i_end = float(np.asarray(res["comp"].select(state["I"]).values.data)[-1, 0])
assert i_end < i_mid
plot_comp(res, "Time-varying recovery rate")


## Infection density flow

$$\lambda = c \, I,\qquad \text{flow} = \lambda \, S.$$


In [ ]:
m = FlowModel(pmap)
mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
m.add_flow(TransitionFlow("infection", state["S"], state["I"], ForceOfInfection("infection", infectious=state["I"], group_by=mixing.prop, mixing=mixing, kind="density", contact_rate=1e-3)))
y0 = jnp.zeros(pmap.size)
y0 = y0.at[pmap.select(state["S"])].set(990.0)
y0 = y0.at[pmap.select(state["I"])].set(10.0)
res = run(m, wrap_y0(y0))
assert float(np.max(np.asarray(res["comp"].select(state["I"]).values.data))) > 10.0
plot_comp(res, "Density-dependent infection")


## Infection frequency flow

$$\lambda = c \, I / N,\qquad \text{flow} = \lambda \, S.$$


In [ ]:
m = FlowModel(pmap)
mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
m.add_flow(TransitionFlow("infection", state["S"], state["I"], ForceOfInfection("infection", infectious=state["I"], group_by=mixing.prop, mixing=mixing, kind="frequency", contact_rate=Param("contact_rate"))))
y0 = jnp.zeros(pmap.size)
y0 = y0.at[pmap.select(state["S"])].set(990.0)
y0 = y0.at[pmap.select(state["I"])].set(10.0)
y0_freq = wrap_y0(y0)
cm_freq = m.compile()
plan_freq = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=jnp.linspace(0.0, 20.0, 201),
)
res = cm_freq.run(
    {"contact_rate": 1.0}, y0_freq, t0=0.0, t1=20.0, dt=0.1, save=plan_freq, solver="euler"
)
assert float(np.max(np.asarray(res["comp"].select(state["I"]).values.data))) > 10.0
plot_comp(res, "Frequency-dependent infection")


## Death flow

`ExitFlow` removes people from a chosen source. Save flow mass to inspect
deaths by cause.


In [ ]:
m = FlowModel(pmap)
m.add_flow(ExitFlow("infection_death", state["I"], 0.03))
m.add_flow(ExitFlow("tiger_death", state["S"], 0.01))
y0 = jnp.zeros(pmap.size)
y0 = y0.at[pmap.select(state["S"])].set(1000.0)
y0 = y0.at[pmap.select(state["I"])].set(1000.0)
res = run(
    m,
    wrap_y0(y0),
    extra={
        "infection_death": SaveRequest(FlowMass(flow="infection_death")),
        "tiger_death": SaveRequest(FlowMass(flow="tiger_death")),
    },
)
assert float(np.asarray(res["comp"].total().values)[-1]) < 2000.0
plot_comp(res, "Two death flows")


In [ ]:
pd.DataFrame(
    {
        "infection_death": np.asarray(res["infection_death"].total().values).ravel(),
        "tiger_death": np.asarray(res["tiger_death"].total().values).ravel(),
    },
    index=np.asarray(res["infection_death"].times.values),
).plot(title="Death flow rates", labels={"index": "time (days)", "value": "people / day"})


## Universal death flow

One `ExitFlow` on every compartment (`Everything()` / all disease states) for
non-disease mortality.


In [ ]:
m = FlowModel(pmap)
m.add_flow(ExitFlow("universal_death", Everything(), 0.02))
y0 = jnp.zeros(pmap.size)
y0 = y0.at[pmap.select(state["S"])].set(700.0)
y0 = y0.at[pmap.select(state["I"])].set(200.0)
y0 = y0.at[pmap.select(state["R"])].set(100.0)
res = run(m, wrap_y0(y0))
assert float(np.asarray(res["comp"].total().values)[-1]) < 1000.0
plot_comp(res, "Universal death")


## Crude birth flow

`EntryFlow` is absolute. Scale a death flux (or a `Reduce`) so births equal
`birth_rate * N` and split across destination strata.


In [ ]:
m = FlowModel(pmap)
# Absolute crude birth: birth_rate * initial N people/day into S.
# (A live N-proportional form is EntryFlow(..., death.sum() * (birth_rate / death_rate)).)
m.add_flow(EntryFlow("birth", state["S"], 0.05 * 1800.0))
y0 = jnp.zeros(pmap.size)
y0 = y0.at[pmap.select(state["S"])].set(700.0)
y0 = y0.at[pmap.select(state["I"])].set(600.0)
y0 = y0.at[pmap.select(state["R"])].set(500.0)
res = run(m, wrap_y0(y0))
assert float(np.asarray(res["comp"].total().values)[-1]) > 1800.0
plot_comp(res, "Crude birth into S")


## Importation flow

Absolute arrivals per day. With a stratified dest, `EntryFlow` splits the
count across matching compartments unless `split=` weights are given.


In [ ]:
m = FlowModel(pmap)
m.add_flow(EntryFlow("imports_S", state["S"], 12.0))
m.add_flow(EntryFlow("imports_I", state["I"], 6.0))
y0 = jnp.zeros(pmap.size)
y0 = y0.at[pmap.select(state["S"])].set(700.0)
y0 = y0.at[pmap.select(state["I"])].set(600.0)
y0 = y0.at[pmap.select(state["R"])].set(500.0)
res = run(m, wrap_y0(y0))
assert float(np.asarray(res["comp"].select(state["S"]).values.data)[-1, 0]) > 700.0
plot_comp(res, "Importation into S and I")


## Replacement birth flow

Births equal total deaths so population is conserved (`death.sum()` into S).


In [ ]:
m = FlowModel(pmap)
death = m.add_flow(ExitFlow("infection_death", state["I"], 0.05))
m.add_flow(EntryFlow("births", state["S"], death.sum()))
y0 = jnp.zeros(pmap.size)
y0 = y0.at[pmap.select(state["S"])].set(650.0)
y0 = y0.at[pmap.select(state["I"])].set(600.0)
res = run(m, wrap_y0(y0))
totals = np.asarray(res["comp"].total().values).ravel()
np.testing.assert_allclose(totals[-1], totals[0], rtol=1e-3)
plot_comp(res, "Replacement births (population conserved)")


## Under `jax.jit`

Differentiate through the frequency-dependent contact rate from the worked
infection example above.


In [ ]:
def loss(contact_rate):
    res = cm_freq.run(
        {"contact_rate": contact_rate},
        y0_freq,
        t0=0.0,
        t1=20.0,
        dt=0.1,
        save=plan_freq,
        solver="euler",
    )
    return jnp.sum(jnp.asarray(res["comp"].select(state["I"]).values.data))


jitted = jax.jit(loss)
val = jitted(jnp.asarray(1.0))
grad = jax.grad(loss)(jnp.asarray(1.0))
assert jnp.isfinite(val) and jnp.isfinite(grad)
np.testing.assert_allclose(val, loss(jnp.asarray(1.0)), rtol=1e-4)
print(f"loss={float(val):.4g}, grad={float(grad):.4g}")


## Summary

| summer2 | summer4 |
|---|---|
| `add_transition_flow` | `TransitionFlow` |
| `add_infection_*_flow` | `TransitionFlow` + `ForceOfInfection` |
| `add_death_flow` / universal | `ExitFlow` |
| `add_crude_birth_flow` / importation / replacement | `EntryFlow` (+ `death.sum()` for replacement) |
| time-varying rates | `piecewise` / `linear` / `Time` |
